# 02 - Model comparison: EfficientNet-B0 vs ResNet50 vs MobileNetV2
All three ImageNet-pretrained backbones are trained with the **same data, splits, augmentation,
class weights and two-phase schedule**:

| Phase | Backbone | Learning rate | Epochs (max) |
|---|---|---|---|
| 1. Feature extraction | frozen | 1e-3 | 8 |
| 2. Fine-tuning | top 40 layers unfrozen (BatchNorm frozen) | 1e-4 | 12 |

Early stopping on validation accuracy, then evaluation on the held-out **test** split.
Besides accuracy we report **macro-F1** (every species counts equally), **top-3 accuracy**
(the system *recommends* candidates for expert review), and **threatened recall**.

> Runtime: ~10-15 min per model on a Colab T4 GPU. *Runtime -> Change runtime type -> T4 GPU*.

In [ ]:
# --- Environment setup: works on Google Colab and locally -------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MaryCassieOdie/MSDS688-Case-Study.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/MSDS688-Case-Study"
    if not os.path.exists(REPO_DIR):
        # If the repo is private, upload the project folder to Drive instead and set
        # REPO_DIR = "/content/drive/MyDrive/MSDS688-Case-Study"
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    # Data and outputs live on Drive so they survive Colab runtime resets.
    DRIVE_ROOT = "/content/drive/MyDrive/MSDS688-Case-Study"
    os.environ["PROJECT_DATA_DIR"] = f"{DRIVE_ROOT}/data"
    os.environ["PROJECT_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
else:
    REPO_DIR = os.path.abspath("..")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
# Optional: official IUCN Red List API token (otherwise iNaturalist's IUCN mirror is used)
# os.environ["IUCN_API_TOKEN"] = "..."   # on Colab, prefer: from google.colab import userdata

from pollinator_id import config
config.ensure_dirs()
print("data   ->", config.DATA_DIR)
print("output ->", config.OUTPUT_DIR)

In [ ]:
# Reading thousands of small files from Google Drive is slow; copy them to the
# Colab VM's local disk once per session for fast training.
import shutil
from pathlib import Path
IMAGE_DIR = config.IMAGE_DIR
if IN_COLAB:
    local = Path("/content/images")
    if not local.exists():
        shutil.copytree(config.IMAGE_DIR, local)
    IMAGE_DIR = local
print("images from", IMAGE_DIR)

In [ ]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from pollinator_id import dataset, models, evaluation
from pollinator_id.config import CLASS_NAMES

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
tf.keras.utils.set_random_seed(config.SEED)

meta = pd.read_csv(config.METADATA_CSV)
iucn_table = pd.read_csv(config.IUCN_CSV)
iucn_codes = dict(zip(iucn_table.scientific_name, iucn_table.iucn_code))
cw = dataset.class_weights(meta)
print(meta.split.value_counts().to_dict())
print({CLASS_NAMES[k]: round(v, 2) for k, v in cw.items()})

In [ ]:
BACKBONES = ["EfficientNetB0", "ResNet50", "MobileNetV2"]
RETRAIN = False  # set True to retrain even if a saved model exists

results, histories, test_probs = [], {}, {}
for name in BACKBONES:
    print(f"\n========== {name} ==========")
    prep = models.preprocess_fn(name)
    train_ds = dataset.make_dataset(meta, "train", prep, image_dir=IMAGE_DIR)
    val_ds = dataset.make_dataset(meta, "val", prep, image_dir=IMAGE_DIR)
    test_ds = dataset.make_dataset(meta, "test", prep, image_dir=IMAGE_DIR)

    ckpt = config.MODEL_DIR / f"{name}.keras"
    hist_file = config.MODEL_DIR / f"{name}_history.json"
    time_file = config.MODEL_DIR / f"{name}_train_seconds.txt"
    if ckpt.exists() and hist_file.exists() and not RETRAIN:
        model = models.load_model(name)
        history = json.loads(hist_file.read_text())
        seconds = float(time_file.read_text()) if time_file.exists() else float("nan")
        print("loaded saved model")
    else:
        model, history, seconds = models.train_two_phase(name, train_ds, val_ds, class_weight=cw)
        time_file.write_text(str(seconds))

    y_true, probs = models.predict_dataset(model, test_ds)
    ms = models.inference_ms_per_image(model)
    results.append(evaluation.summarize(name, y_true, probs, model.count_params(), seconds, ms,
                                        iucn_codes=iucn_codes))
    histories[name], test_probs[name] = history, (y_true, probs)
    np.savez(config.MODEL_DIR / f"{name}_test_predictions.npz", y_true=y_true, probs=probs)

res = pd.DataFrame(results).set_index("model")
res.to_csv(config.OUTPUT_DIR / "model_comparison.csv")
res.style.format("{:.3f}").highlight_max(subset=["accuracy", "top3_accuracy", "macro_f1", "threatened_recall"],
                                         color="#c7e9c0")

## Learning curves

In [ ]:
fig = evaluation.plot_histories(histories)
fig.savefig(config.FIGURE_DIR / "learning_curves.png", dpi=150)

## Accuracy vs. cost

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, r in res.iterrows():
    axes[0].scatter(r.params_millions, r.macro_f1, s=120); axes[0].annotate(name, (r.params_millions, r.macro_f1), xytext=(6, 4), textcoords="offset points")
    axes[1].scatter(r.ms_per_image, r.macro_f1, s=120); axes[1].annotate(name, (r.ms_per_image, r.macro_f1), xytext=(6, 4), textcoords="offset points")
axes[0].set_xlabel("parameters (millions)"); axes[1].set_xlabel("inference ms / image (batch 32)")
for a in axes: a.set_ylabel("test macro-F1"); a.grid(alpha=.3)
fig.suptitle("Model size and speed vs. accuracy"); fig.tight_layout()
fig.savefig(config.FIGURE_DIR / "accuracy_vs_cost.png", dpi=150)

## Confusion matrices (row-normalised, test split)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5))
for ax, name in zip(axes, BACKBONES):
    evaluation.plot_confusion(*test_probs[name], title=name, ax=ax)
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / "confusion_matrices.png", dpi=130)

## Per-species F1 - where do the models differ?

In [ ]:
f1 = pd.DataFrame({n: evaluation.per_class_report(*test_probs[n]).loc[CLASS_NAMES, "f1-score"] for n in BACKBONES})
ax = f1.plot.barh(figsize=(10, 7), width=.8)
ax.set_yticklabels([f"{n} [{iucn_codes[n]}]" for n in f1.index]); ax.invert_yaxis()
ax.set_xlabel("F1 (test)"); ax.set_title("Per-species F1 by backbone (IUCN category in brackets)"); ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.savefig(config.FIGURE_DIR / "per_species_f1.png", dpi=150)
f1.style.format("{:.2f}").background_gradient(cmap="Greens", axis=None)

## Select the model used downstream
The deployed system **recommends** species to human experts, and the costliest error is missing a
threatened species. We therefore pick the model with the best **macro-F1**, breaking ties by
**threatened recall**. Change `SELECTION_METRIC` to choose differently (e.g. `ms_per_image` for a
mobile/field deployment, where MobileNetV2 is designed to shine).

In [ ]:
SELECTION_METRIC = "macro_f1"
best = res.sort_values([SELECTION_METRIC, "threatened_recall"], ascending=False).index[0]
(config.OUTPUT_DIR / "best_model.json").write_text(json.dumps({"backbone": best, "selected_by": SELECTION_METRIC}))
print("Selected backbone:", best)
evaluation.per_class_report(*test_probs[best]).round(3)

### Discussion prompts
* Which bumble-bee pairs are most confused (confusion matrices)? Are they colour-pattern look-alikes,
  e.g. *B. affinis* vs. *B. griseocollis* / *B. impatiens*?
* Does the larger ResNet50 actually beat the far smaller EfficientNet-B0 / MobileNetV2?
* How large is the gap between top-1 and top-3 accuracy? A big gap supports a
  "shortlist for expert review" product rather than an automatic ID.
* Compare `threatened_flag_recall` (right *status*, even if wrong species) with `threatened_recall`.